# QDAC sourced NI DAQ CrossTalk Test

## Import Libraries

In [1]:
import time
import json
import pyvisa
import numpy as np
import matplotlib.pyplot as plt
from time import sleep

from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qcodes.instrument.channel import ChannelList

from qstl_instruments.qstl_qdac2 import QSTL_QDac2
from qstl_instruments.qstl_nidaq import QSTL_NIDaq

from pyvisa.constants import StopBits, Parity

rm = pyvisa.ResourceManager()
rm.list_resources()

('ASRL1::INSTR',
 'ASRL3::INSTR',
 'ASRL5::INSTR',
 'ASRL20::INSTR',
 'ASRL21::INSTR',
 'ASRL22::INSTR',
 'ASRL23::INSTR',
 'ASRL24::INSTR',
 'ASRL25::INSTR',
 'ASRL26::INSTR',
 'ASRL27::INSTR')

## Instantiation of Instruments

In [3]:
contacts = {
    "X" : 1,
    "Y" : 2
}

ai_chans = {
    "I" : "Dev2/ai0",
    "Q" : "Dev2/ai2",
}

# Set up database
initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251009_QDAC_NIQAC_synctest/QDAC_IQ_Mod_Bandwidth_Test.db")

qdac2 = QSTL_QDac2(
    name = "qdac2",
    address = "ASRL5::INSTR",
    ramp_rate = 1,
    i_threshold = 2e-9,
    v_limit = 0.5,
    contacts = contacts
)
station = Station(qdac2)

daq = QSTL_NIDaq(
    max_sampling_rate = int(1e6),
    gain = 1e8
)

qdac2.ramp_all_channels_to_zero()
qdac2.get_initial_voltages()

Connected to: QDevil QDAC-II (serial:368, firmware:13-1.57) in 0.05s


{'X': 0.0, 'Y': 0.0}

## Instruments Setup

In [11]:
## setup qdac 2 for the 2D sweep
qdac2.free_all_triggers()
qdac2.ext3.delay_s(0)
qdac2.v_limit = 1.2

device1 = "I"
device2 = "Q"

slow_chans = ["X"]
fast_chans = ["Y"]

slow_start = 0.0
slow_end = 0.2
slow_steps = 100

fast_start = 0.0
fast_end = 0.2
fast_steps = 100
fast_step_time_s = 2e-6

qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")

for i in [slow_start, slow_end, fast_start, fast_end]:
    qdac2.validate_voltages([i])

slow_vs = np.linspace(slow_start, slow_end, slow_steps, endpoint=True)
fast_vs = np.linspace(fast_start, fast_end, fast_steps, endpoint=True)

exp = load_or_create_experiment("2D sweep", "QDAC+NiDAQ_Xtalk_Test")
meas = Measurement(exp=exp, station=station)

Vslow = Parameter(name="Vslow", label=str(slow_chans), unit="V")
Vfast = Parameter(name="Vfast", label=str(fast_chans), unit="V")

I_multi = Parameter(name= "I_multi", label="I_multi", unit="V")
Q_multi = Parameter(name= "Q_multi", label="Q_multi", unit="V")

A_multi = Parameter(name= "A_multi", label="A_multi", unit="V")
Phi_multi = Parameter(name= "theta_multi", label="theta_multi", unit="deg")

I_single = Parameter(name= "Isingle", label="I_single", unit="V")
Q_single = Parameter(name= "Q_single", label="Q_single", unit="V")

A_single = Parameter(name= "A_single", label="A_single", unit="V")
Phi_single = Parameter(name= "theta_single", label="theta_single", unit="deg")

meas.register_parameter(Vfast)
meas.register_parameter(Vslow)
meas.register_parameter(I_multi, setpoints = (Vfast, Vslow))
meas.register_parameter(Q_multi, setpoints = (Vfast, Vslow))
meas.register_parameter(A_multi, setpoints = (Vfast, Vslow))
meas.register_parameter(Phi_multi, setpoints = (Vfast, Vslow))
meas.register_parameter(I_single, setpoints = (Vfast, Vslow))
meas.register_parameter(Q_single, setpoints = (Vfast, Vslow))
meas.register_parameter(A_single, setpoints = (Vfast, Vslow))
meas.register_parameter(Phi_single, setpoints = (Vfast, Vslow))

## Multi Channel Measurement

In [14]:
qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")
qdac2.free_all_triggers()

sweep_time = fast_steps * fast_step_time_s
samples_per_fast_scan_per_channel = fast_steps * int(fast_step_time_s*daq.max_sampling_rate/2)

arrangement = qdac2.arrange(
    contacts= {x: contacts[x] for x in fast_chans},
    output_triggers={
        "NIDAQ" : 5,
    }
)

sweep = arrangement.virtual_detune(
    contacts = tuple(fast_chans),
    start_V = (fast_start,) * len(fast_chans),
    end_V = (fast_end,) * len(fast_chans),
    steps = fast_steps,
    step_trigger = "NIDAQ",
    step_time_s = fast_step_time_s,
    repetitions = 1
)

InitialConditions = qdac2.get_initial_voltages()

start_time = time.time()
with meas.run() as datasaver:
    loop_counter = 0
    datasaver.dataset.add_metadata(tag="Contacts", metadata=json.dumps(contacts))
    datasaver.dataset.add_metadata(tag="IC", metadata=json.dumps(InitialConditions))
    datasaver.dataset.add_metadata(
        tag = "Sweep_params",
        metadata = json.dumps(
            {
                "slow_chans" : slow_chans,
                "slow_start" : slow_start,
                "slow_end" : slow_end,
                "fast_chans" : fast_chans,
                "fast_start" : fast_start,
                "fast_end" : fast_end,
                "fast_step_time_s" : fast_step_time_s
            }
        )
    )
    for slow_v in slow_vs:
        qdac2.ramp_channels(slow_chans, [slow_v])
        # Read NI Daq traces
        result = daq.read_triggered_multi_channels(
            sweep,
            [ai_chans[device1], ai_chans[device2]],
            samples_per_fast_scan_per_channel,
            -1,
            +1,
            sweep_time+1
        )
        # Read QCS Digitizer Traces

        # Read NI DAQ Samples
        result_0 = daq.reshape_array(result[0,:], fast_steps)
        result_1 = daq.reshape_array(result[1,:], fast_steps)

        amp = np.abs(result_0+1j*result_1)
        phase = np.angle(result_0+1j*result_1, deg=True)

        datasaver.add_result(
            (Vslow, [slow_v]*fast_steps),
            (Vfast, fast_vs),
            (I_multi, result_0),
            (Q_multi, result_1),
            (A_multi, amp),
            (Phi_multi, phase),
        )

        result_0 = daq.read_triggered_voltage(sweep, ai_chans[device1], samples_per_fast_scan_per_channel << 1, -1, +1, sweep_time+1)
        result_0 = daq.reshape_array(result_0, fast_steps)

        result_1 = daq.read_triggered_voltage(sweep, ai_chans[device2], samples_per_fast_scan_per_channel << 1, -1, +1, sweep_time+1)
        result_1 = daq.reshape_array(result_1, fast_steps)

        amp = np.abs(result_0+1j*result_1)
        phase = np.angle(result_0+1j*result_1, deg=True)

        datasaver.add_result(
            (Vslow, [slow_v]*fast_steps),
            (Vfast, fast_vs),
            (I_single, result_0),
            (Q_single, result_1),
            (A_single, amp),
            (Phi_single, phase),
        )

        loop_counter = loop_counter+1
        print(f'Time elapsed: {np.round(time.time()-start_time, 2)} sec. Loop finished: {loop_counter}/{slow_steps}.')

end_time = time.time()
print(f'Time elapsed: {np.round(end_time-start_time, 2)} sec.')

qdac2.channels[0:23].dc_slew_rate_V_per_s(1)

qdac2.ramp_all_channels_to_zero()

Starting experimental run with id: 143. 
Time elapsed: 0.12 sec. Loop finished: 1/100.
Time elapsed: 0.18 sec. Loop finished: 2/100.
Time elapsed: 0.24 sec. Loop finished: 3/100.
Time elapsed: 0.32 sec. Loop finished: 4/100.
Time elapsed: 0.36 sec. Loop finished: 5/100.
Time elapsed: 0.43 sec. Loop finished: 6/100.
Time elapsed: 0.49 sec. Loop finished: 7/100.
Time elapsed: 0.55 sec. Loop finished: 8/100.
Time elapsed: 0.61 sec. Loop finished: 9/100.
Time elapsed: 0.68 sec. Loop finished: 10/100.
Time elapsed: 0.74 sec. Loop finished: 11/100.
Time elapsed: 0.8 sec. Loop finished: 12/100.
Time elapsed: 0.86 sec. Loop finished: 13/100.
Time elapsed: 0.91 sec. Loop finished: 14/100.
Time elapsed: 0.96 sec. Loop finished: 15/100.
Time elapsed: 1.0 sec. Loop finished: 16/100.
Time elapsed: 1.05 sec. Loop finished: 17/100.
Time elapsed: 1.11 sec. Loop finished: 18/100.
Time elapsed: 1.17 sec. Loop finished: 19/100.
Time elapsed: 1.24 sec. Loop finished: 20/100.
Time elapsed: 1.3 sec. Loop fi

## QDAC2 reset

In [15]:
qdac2.reset()